# Length Helper
Планировщик производства кабеля: изолирование → скрутка → намотка на барабаны.

**Использование:**
1. Заполните `input.xlsx` (заказы, склад, ёмкости барабанов, параметры).
2. Укажите пути к файлам в ячейке «Настройки».
3. Запустите все ячейки (Kernel → Restart & Run All).

In [ ]:
# ─── Настройки ───────────────────────────────────────────────────────
# INPUT_FILE  = 'input.xlsx'
INPUT_FILE  = 'input26.xlsx'
OUTPUT_FILE = 'output26.xlsx'

In [38]:
import sys, importlib
# Перезагружаем модули при повторных запусках
for mod in ['models', 'parser', 'planner', 'exporter']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from parser  import parse_input
from planner import plan
from exporter import export

In [39]:
# ─── Чтение входного файла ───────────────────────────────────────────
orders, raw_wires, insulated_cores, cable_stock, \
    core_drum_caps, cable_drum_caps, params = parse_input(INPUT_FILE)

print(f'Заказов:          {len(orders)}')
print(f'Барабанов ТПЖ:    {len(raw_wires)}')
print(f'Жил на складе:    {len(insulated_cores)}')
print(f'Кабеля на складе: {len(cable_stock)}')
print(f'Сечений жил:      {len(core_drum_caps)}')
print(f'Марок кабелей:    {len(cable_drum_caps)}')
print()
print(f'Макс. изолирование:   {params.max_insulation_run} м')
print(f'Макс. скрутка:        {params.max_twisting_run} м')
print(f'Мин. строит. длина:   {params.min_construction_length} м')
print(f'Спайка разрешена:     {params.allow_splicing}')

⚠ Марка "КОСМОГЕР Вз-ВБШвнг(А)-LS 3х2,5ок(N,PE)-1  ТУ 27.32.13-004-32408375-2022" не найдена в листе "Состав кабелей" — цвета и сечение не определены.
⚠ Марка "КОСМОГЕР Вз-ВБШвнг(А)-LS 3х2,5ок(N,PE)-1  ТУ 27.32.13-004-32408375-2022" не найдена в листе "Состав кабелей" — цвета и сечение не определены.
⚠ Марка "КОСМОГЕР Вз-ВБШвнг(А)-LS 3х25мк(N,PE)-1  ТУ 27.32.13-004-32408375-2022" не найдена в листе "Состав кабелей" — цвета и сечение не определены.
⚠ Марка "КОСМОГЕР Вз-ВБШвнг(А)-LS 3х4ок(N,PE)-1  ТУ 27.32.13-004-32408375-2022" не найдена в листе "Состав кабелей" — цвета и сечение не определены.
⚠ Марка "КОСМОГЕР Вз-ВБШвнг(А)-LS 3х4ок(N,PE)-1  ТУ 27.32.13-004-32408375-2022" не найдена в листе "Состав кабелей" — цвета и сечение не определены.
Заказов:          5
Барабанов ТПЖ:    14
Жил на складе:    4
Кабеля на складе: 2
Сечений жил:      4
Марок кабелей:    5

Макс. изолирование:   4500.0 м
Макс. скрутка:        2100.0 м
Мин. строит. длина:   450.0 м
Спайка разрешена:     False


In [40]:
# ─── Планирование ────────────────────────────────────────────────────
result = plan(
    orders, raw_wires, insulated_cores, cable_stock,
    core_drum_caps, cable_drum_caps, params
)

print(f'Партий скрутки:       {len(result.batches)}')
print(f'Прогонов изолирования:{len(result.insulation_runs)}')
print(f'Жил со склада:        {len(result.insulated_core_uses)}')
print(f'Кабеля со склада:     {len(result.cable_stock_uses)}')
print(f'Выходных барабанов:   {len(result.drum_assignments)}')
print()
if result.errors:
    print('=== ОШИБКИ ===')
    for e in result.errors:
        print(' ', e)
if result.warnings:
    print('=== ПРЕДУПРЕЖДЕНИЯ ===')
    for w in result.warnings:
        print(' ', w)

Партий скрутки:       0
Прогонов изолирования:0
Жил со склада:        0
Кабеля со склада:     0
Выходных барабанов:   0

=== ОШИБКИ ===
  ❌ "КОСМОГЕР Вз-ВБШвнг(А)-LS 3х2,5ок(N,PE)-1  ТУ 27.32.13-004-32408375-2022": состав (цвета жил) не определён — пропускаем заказ.
  ❌ "КОСМОГЕР Вз-ВБШвнг(А)-LS 3х2,5ок(N,PE)-1  ТУ 27.32.13-004-32408375-2022": состав (цвета жил) не определён — пропускаем заказ.
  ❌ "КОСМОГЕР Вз-ВБШвнг(А)-LS 3х25мк(N,PE)-1  ТУ 27.32.13-004-32408375-2022": состав (цвета жил) не определён — пропускаем заказ.
  ❌ "КОСМОГЕР Вз-ВБШвнг(А)-LS 3х4ок(N,PE)-1  ТУ 27.32.13-004-32408375-2022": состав (цвета жил) не определён — пропускаем заказ.
  ❌ "КОСМОГЕР Вз-ВБШвнг(А)-LS 3х4ок(N,PE)-1  ТУ 27.32.13-004-32408375-2022": состав (цвета жил) не определён — пропускаем заказ.


In [41]:
# ─── Инструкция по скрутке ───────────────────────────────────────────
import pandas as pd
from collections import defaultdict

# Индексы: партия → [InsulationRun], [InsulatedCoreUse], [DrumAssignment]
ins_run_by_batch  = defaultdict(list)
ins_use_by_batch  = defaultdict(list)
drum_by_batch     = defaultdict(list)

for r in result.insulation_runs:
    ins_run_by_batch[r.for_batch_id].append(r)
for u in result.insulated_core_uses:
    ins_use_by_batch[u.for_batch_id].append(u)
for da in result.drum_assignments:
    if da.source == 'партия' and da.batch_id:
        drum_by_batch[da.batch_id].append(da)

for batch in result.batches:
    bid = batch.id
    segs_str = ' + '.join(str(int(s)) for s in batch.segments)
    fr_info  = f'  FR: {batch.fire_resistant}' if batch.fire_resistant else ''
    print('═' * 72)
    print(f'ПАРТИЯ {bid}:  {batch.cable_mark}')
    print(f'  Тип жилы: {batch.wire_key} | Материал: {batch.insulation_material}{fr_info}')
    print(f'  Отрезки:  {segs_str} = {int(batch.total_length)} м суммарно')
    print()

    # ── ЗАПРАВКА МАШИНЫ ───────────────────────────────────────────────
    print('  ЗАПРАВКА МАШИНЫ СКРУТКИ:')
    setup_rows = []
    for pos, color in enumerate(batch.colors, 1):
        run = next((r for r in ins_run_by_batch[bid] if r.color == color), None)
        use = next((u for u in ins_use_by_batch[bid] if u.color == color), None)

        if run:
            src_type = 'Прогон изолирования'
            src_name = f'{run.id}  (барабан/моток: {run.drum_type})'
            length_m = int(run.length)
        elif use:
            src_type = 'Готовая жила (склад)'
            src_name = use.source_name
            length_m = int(use.length)
        else:
            src_type = '— не определено —'
            src_name = ''
            length_m = 0

        setup_rows.append({
            'Поз.':    pos,
            'Цвет':    color,
            'Откуда':  src_type,
            'Источник': src_name,
            'Длина, м': length_m,
        })

    df_setup = pd.DataFrame(setup_rows).set_index('Поз.')
    display(df_setup)
    print()

    # ── ПОСЛЕДОВАТЕЛЬНОСТЬ ОТРЕЗКОВ И БАРАБАНЫ ────────────────────────
    print('  ПОСЛЕДОВАТЕЛЬНОСТЬ ОТРЕЗКОВ И ПРИЁМНЫЕ БАРАБАНЫ:')

    # Для каждого отрезка определяем, на какой барабан он идёт.
    # DrumAssignment.segments содержит длины в порядке добавления.
    # Копируем список сегментов каждого барабана и «тратим» их по очереди.
    drum_seg_queues = {
        da.id: list(da.segments) for da in drum_by_batch[bid]
    }
    drum_list = list(drum_by_batch[bid])  # упорядоченный список барабанов партии

    seq_rows = []
    cumulative = 0
    for step, seg in enumerate(batch.segments, 1):
        cumulative += seg
        # Найти барабан, у которого следующий неиспользованный сегмент совпадает
        matched_da = None
        for da in drum_list:
            q = drum_seg_queues[da.id]
            if q and abs(q[0] - seg) < 0.5:
                matched_da = da
                q.pop(0)
                break
        # Если не нашли точного совпадения — ищем любой барабан с этой длиной
        if matched_da is None:
            for da in drum_list:
                q = drum_seg_queues[da.id]
                for i, v in enumerate(q):
                    if abs(v - seg) < 0.5:
                        matched_da = da
                        q.pop(i)
                        break
                if matched_da:
                    break

        seq_rows.append({
            'Шаг':          step,
            'Длина, м':     int(seg),
            'Нарастающим, м': int(cumulative),
            'Барабан':      matched_da.id if matched_da else '—',
            'Тип барабана': matched_da.drum_type if matched_da else '—',
            'Ёмк., м':     int(matched_da.drum_capacity) if matched_da else 0,
        })

    df_seq = pd.DataFrame(seq_rows).set_index('Шаг')
    display(df_seq)
    print()

print('═' * 72)

════════════════════════════════════════════════════════════════════════


In [42]:
# ─── Инструкция по изолированию ──────────────────────────────────────
import pandas as pd

batch_mark  = {b.id: b.cable_mark  for b in result.batches}
batch_total = {b.id: b.total_length for b in result.batches}

# Начальные остатки ТПЖ-барабанов (из оригинального списка до планирования)
tpzh_avail = {rw.id: rw.length for rw in raw_wires}

# Начальные остатки изолированных жил на складе (до планирования)
inscore_avail = {ic.id: ic.length for ic in insulated_cores}

# Объединяем прогоны изолирования и использования со склада,
# сортируем по партии скрутки, затем по цвету — это рабочая последовательность
all_ops = sorted(
    [('ТПЖ', r) for r in result.insulation_runs] +
    [('СКЛ', u) for u in result.insulated_core_uses],
    key=lambda x: (x[1].for_batch_id, x[1].color)
)

rows = []
for n, (op_type, item) in enumerate(all_ops, 1):
    bid   = item.for_batch_id
    mark  = batch_mark.get(bid, '?')
    btot  = int(batch_total.get(bid, 0))

    if op_type == 'ТПЖ':
        run = item
        # raw_wire_consumed = длина прогона + потери на заправку + запас на обрезку торцов
        consumed = run.raw_wire_consumed if run.raw_wire_consumed > 0 else run.length
        before = round(tpzh_avail.get(run.source_id, 0))
        tpzh_avail[run.source_id] = tpzh_avail.get(run.source_id, 0) - consumed
        after = round(tpzh_avail[run.source_id])
        mat = run.insulation_material + (f'+{run.fire_resistant}' if run.fire_resistant else '')
        rows.append({
            '№':                  n,
            'Тип операции':       'Прогон (ТПЖ → изол.)',
            'Партия':             bid,
            'Марка кабеля':       mark,
            'Партия, м':          btot,
            'Цвет жилы':          run.color,
            'Тип жилы':           run.wire_key,
            'Материал/FR':        mat,
            'Источник':           run.source_name,
            'Остаток ДО, м':      before,
            'Длина прогона, м':   int(run.length),
            'Снять с барабана, м': int(consumed),
            'Остаток ПОСЛЕ, м':   after,
            'Принять на':         run.drum_type,
        })
    else:
        use = item
        before = round(inscore_avail.get(use.source_id, 0))
        inscore_avail[use.source_id] = use.remainder
        rows.append({
            '№':                  n,
            'Тип операции':       'Склад (готовая жила)',
            'Партия':             bid,
            'Марка кабеля':       mark,
            'Партия, м':          btot,
            'Цвет жилы':          use.color,
            'Тип жилы':           use.wire_key,
            'Материал/FR':        '(со склада)',
            'Источник':           use.source_name,
            'Остаток ДО, м':      before,
            'Длина прогона, м':   int(use.length),
            'Снять с барабана, м': '—',
            'Остаток ПОСЛЕ, м':   int(use.remainder),
            'Принять на':         '(уже готовая)',
        })

pd.DataFrame(rows)

""


In [43]:
# ─── Потребность в полуфабрикатах: ТПЖ и FR-жила ─────────────────────
#
# Логика:
#   • Стандартный прогон  → ТПЖ берётся как есть
#   • FR-прогон           → ТПЖ требует предварительной операции
#                           «нанесение огнестойкой обмотки» (или закупается
#                           ТПЖ FR как отдельный полуфабрикат)
#   • Ошибки планировщика → дефицит: жила не была выделена → требует заказа
# ─────────────────────────────────────────────────────────────────────
import pandas as pd
from collections import defaultdict

batch_mark = {b.id: b.cable_mark for b in result.batches}

# ── Потребность из успешно спланированных прогонов ───────────────────
plain_need: dict = defaultdict(float)   # wire_key → м стандарт
fr_need:    dict = defaultdict(float)   # wire_key → м с FR-обмоткой

for run in result.insulation_runs:
    if run.fire_resistant == 'FR':
        fr_need[run.wire_key] += run.length
    else:
        plain_need[run.wire_key] += run.length

# ── Дефицит: разбираем ошибки планировщика ───────────────────────────
# Ошибки вида «нужно X м ТПЖ Y, доступно Z м» — недостаток на складе
import re as _re

deficit_plain: dict = defaultdict(float)
deficit_fr:    dict = defaultdict(float)

for err in result.errors:
    m = _re.search(r'нужно\s+([\d]+)\s+м\s+ТПЖ\s+(\S+),', err)
    if not m:
        continue
    qty     = float(m.group(1))
    wk      = m.group(2)
    is_fr   = '+FR' in err or 'FR]' in err
    if is_fr:
        deficit_fr[wk]    += qty
    else:
        deficit_plain[wk] += qty

# ── Наличие ТПЖ на складе (исходное, до планирования) ────────────────
stock_by_wk: dict = defaultdict(float)
for rw in raw_wires:                    # оригинальный список — не изменён plan()
    stock_by_wk[rw.wire_key] += rw.length

all_wk = sorted(
    set(plain_need) | set(fr_need) | set(deficit_plain) | set(deficit_fr) | set(stock_by_wk)
)

# ══ Таблица 1: Потребность в ТПЖ ════════════════════════════════════
rows = []
for wk in all_wk:
    p     = plain_need.get(wk, 0.0)
    f     = fr_need.get(wk, 0.0)
    dp    = deficit_plain.get(wk, 0.0)
    df    = deficit_fr.get(wk, 0.0)
    total_need = p + f + dp + df
    stock      = stock_by_wk.get(wk, 0.0)
    order_qty  = dp + df                    # сколько нужно докупить/изготовить

    def _fmt(v): return int(v) if v > 0 else '—'
    rows.append({
        'Тип ТПЖ':              wk,
        'На складе, м':         int(stock),
        'Стандарт (план), м':   _fmt(p),
        'FR-обмотка (план), м': _fmt(f),
        'Стандарт (дефицит), м':_fmt(dp),
        'FR-обмотка (дефицит),м':_fmt(df),
        'Итого нужно, м':       int(total_need),
        'Действие':             f'⚠ Заказать {int(order_qty)} м' if order_qty > 0 else '✓ Достаточно',
    })

print('══════  ПОТРЕБНОСТЬ В ТПЖ (ГОЛАЯ ЖИЛА)  ══════\n')
display(pd.DataFrame(rows))

# ══ Таблица 2: Программа FR-обмотки ═════════════════════════════════
all_fr_runs = [r for r in result.insulation_runs if r.fire_resistant == 'FR']

if all_fr_runs or deficit_fr:
    print('\n══════  ПОЛУФАБРИКАТ: ЖИЛА С FR-ОБМОТКОЙ  ══════')
    print('Перед изолированием необходима операция нанесения огнестойкой обмотки (ленты).\n')

    if all_fr_runs:
        fr_rows = []
        for run in all_fr_runs:
            fr_rows.append({
                'Партия':             run.for_batch_id,
                'Марка кабеля':       batch_mark.get(run.for_batch_id, ''),
                'Цвет жилы':          run.color,
                'Тип ТПЖ':            run.wire_key,
                'Материал изол.':     run.insulation_material,
                'Источник (барабан)': run.source_name,
                'Длина прогона, м':   int(run.length),
                'Моток':              run.drum_type,
            })
        display(pd.DataFrame(fr_rows))

        # Сводка по барабанам ТПЖ, задействованным под FR
        print('\nБарабаны ТПЖ под FR-обмотку (нанести обмотку на отмеченные участки):')
        drum_fr: dict = defaultdict(float)
        for run in all_fr_runs:
            drum_fr[(run.wire_key, run.source_name)] += run.length
        for (wk, drum_name), meters in sorted(drum_fr.items()):
            print(f'  ТПЖ {wk:10s}  барабан {drum_name:20s}  → {int(meters):5d} м FR-обмотки')

    if deficit_fr:
        print('\n⚠  Дополнительно требуется (дефицит ТПЖ под FR, не покрытый складом):')
        for wk, qty in sorted(deficit_fr.items()):
            print(f'  ТПЖ {wk}: {int(qty)} м  → заказать или изготовить ТПЖ FR {wk}')
else:
    print('\nFR-жилы в текущей производственной программе отсутствуют.')

══════  ПОТРЕБНОСТЬ В ТПЖ (ГОЛАЯ ЖИЛА)  ══════



,Тип ТПЖ,"На складе, м","Стандарт (план), м","FR-обмотка (план), м","Стандарт (дефицит), м","FR-обмотка (дефицит),м","Итого нужно, м",Действие
0,"2,5мк",27000,—,—,—,—,0,✓ Достаточно
1,"2,5ок",30400,—,—,—,—,0,✓ Достаточно
2,25мк,20000,—,—,—,—,0,✓ Достаточно
3,25мкг,16000,—,—,—,—,0,✓ Достаточно



FR-жилы в текущей производственной программе отсутствуют.


In [44]:
# ─── Выходные барабаны ───────────────────────────────────────────────
rows = []
for da in result.drum_assignments:
    rows.append({
        'ID':         da.id,
        'Марка':      da.cable_mark,
        'Тип':        da.drum_type,
        'Ёмкость, м': int(da.drum_capacity),
        'Отрезки':    ', '.join(str(int(s)) for s in da.segments),
        'Итого, м':   int(da.total_length),
        'Загрузка':   f"{da.total_length/da.drum_capacity*100:.0f}%" if da.drum_capacity else '—',
        'Источник':   da.source,
    })
pd.DataFrame(rows)

""


In [45]:
# ─── Экспорт ─────────────────────────────────────────────────────────
export(OUTPUT_FILE, result)
print('Готово!')

Результат сохранён: output.xlsx
Готово!
